# 311 Service Request Router - Exploratory Data Analysis

This notebook explores the Calgary 311 Service Requests dataset to understand patterns,
distributions, and relationships that will inform our ML routing model.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
from src.data_loader import load_or_fetch_data, preprocess_data, engineer_features

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 1. Load Data

In [ ]:
df_raw = load_or_fetch_data('../data', limit=100000)
print(f'Raw dataset shape: {df_raw.shape}')
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe()

## 2. Data Quality Assessment

In [ ]:
# Missing values
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
missing_df[missing_df['Missing'] > 0].sort_values('Percent', ascending=False)

## 3. Preprocess & Engineer Features

In [ ]:
df = preprocess_data(df_raw)
df = engineer_features(df)
print(f'Processed dataset shape: {df.shape}')
df.head()

## 4. Target Variable Analysis (Department Distribution)

In [ ]:
dept_counts = df['agency_responsible'].value_counts()
fig = px.bar(x=dept_counts.index, y=dept_counts.values,
             title='Request Count by Department (Top 15)',
             labels={'x': 'Department', 'y': 'Request Count'})
fig.update_layout(xaxis_tickangle=-45, height=450)
fig.show()

In [ ]:
print('Department Distribution:')
print(dept_counts)
print(f'\nTotal departments: {len(dept_counts)}')
print(f'Top 5 departments handle {dept_counts.head(5).sum() / len(df) * 100:.1f}% of requests')

## 5. Categorical Feature Analysis

In [ ]:
for col in ['channel', 'status']:
    if col in df.columns:
        counts = df[col].value_counts().head(15)
        fig = px.bar(x=counts.index, y=counts.values,
                     title=f'Distribution of {col}',
                     labels={'x': col, 'y': 'Count'})
        fig.update_layout(height=350)
        fig.show()

In [ ]:
# Service request type distribution
top_types = df['service_request_type'].value_counts().head(20)
fig = px.bar(x=top_types.values, y=top_types.index, orientation='h',
             title='Top 20 Service Request Types',
             labels={'x': 'Count', 'y': 'Service Request Type'})
fig.update_layout(yaxis=dict(autorange='reversed'), height=600)
fig.show()

## 6. Temporal Trends

In [ ]:
if 'year' in df.columns:
    yearly = df.groupby('year').size().reset_index(name='count')

    fig = px.bar(yearly, x='year', y='count',
                 title='Annual Request Volume',
                 labels={'year': 'Year', 'count': 'Request Count'})
    fig.update_layout(height=400)
    fig.show()

if 'hour' in df.columns:
    hourly = df.groupby('hour').size().reset_index(name='count')
    fig = px.bar(hourly, x='hour', y='count',
                 title='Requests by Hour of Day',
                 labels={'hour': 'Hour', 'count': 'Request Count'})
    fig.update_layout(height=350)
    fig.show()

## 7. Resolution Time Analysis

In [ ]:
if 'resolution_hours' in df.columns:
    res_df = df[df['resolution_hours'].between(0, 720)]
    fig = px.histogram(res_df, x='resolution_hours', nbins=50,
                       title='Distribution of Resolution Times (up to 30 days)',
                       labels={'resolution_hours': 'Resolution Hours'})
    fig.update_layout(height=400)
    fig.show()

    print('Resolution Time Statistics (hours):')
    print(df['resolution_hours'].describe())

## 8. Correlation Analysis

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_cols) > 1:
    key_numeric = [c for c in ['hour', 'day_of_week', 'month', 'year',
                   'community_request_count', 'community_avg_resolution',
                   'resolution_hours', 'service_type_frequency']
                   if c in df.columns]
    if len(key_numeric) > 1:
        fig = px.imshow(df[key_numeric].corr(), text_auto='.2f',
                        title='Feature Correlation Heatmap', color_continuous_scale='RdBu_r')
        fig.update_layout(height=500)
        fig.show()

## 9. Key Takeaways

1. **Department distribution is highly imbalanced** - a few departments handle the majority of requests
2. **Service request type is the strongest signal** for department routing
3. **Channel matters** - phone and web dominate, different channels may route differently
4. **Temporal patterns exist** - clear hourly and seasonal trends in request volume
5. **Resolution times vary widely** by department, from hours to weeks
6. **Community-level features** add useful context for routing predictions